In [ ]:
from faker import Faker
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import random
from datetime import datetime
import numpy as np
import os

output_folder = "receipts_straight"
os.makedirs(output_folder, exist_ok=True)

def generate_realistic_receipt_with_all_noises_and_effects(filename="receipt.png", rotate_angle=0, perspective_perturb=0):
    fake = Faker('en_IN')

    img_width = 700
    img_height = 1500
    padding = 40
    background_color = (245, 245, 245)
    text_color = (30, 30, 30)
    img = Image.new('RGB', (img_width, img_height), color=background_color)
    draw = ImageDraw.Draw(img)

    try:
        font_path_regular = "arial.ttf"
        font_path_bold = "arialbd.ttf"
        font_xsmall = ImageFont.truetype(font_path_regular, 17) 
        font_small = ImageFont.truetype(font_path_regular, 19)
        font_medium = ImageFont.truetype(font_path_regular, 22)
        font_large = ImageFont.truetype(font_path_bold, 32)
        font_item_name = ImageFont.truetype(font_path_regular, 19)
        font_total = ImageFont.truetype(font_path_bold, 28)
        font_thankyou = ImageFont.truetype(font_path_regular, 22)
    except IOError:
        font_xsmall = ImageFont.load_default()
        font_small = ImageFont.load_default()
        font_medium = ImageFont.load_default()
        font_large = ImageFont.load_default()
        font_item_name = ImageFont.load_default()
        font_total = ImageFont.load_default()
        font_thankyou = ImageFont.load_default()

    layouts = [
        {'name_align': random.choice(['left', 'right', 'center']),
         'address_align': random.choice(['left', 'right', 'center']),
         'phone_align': random.choice(['left', 'right', 'center']),
         'gst_align': random.choice(['left', 'right', 'center']),
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'left',
         'address_align': 'left',
         'phone_align': 'left',
         'gst_align': 'left',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.45,
         'price_col_x': img_width * 0.58,
         'vat_col_x': img_width * 0.73,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'right',
         'address_align': 'right',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.35,
         'price_col_x': img_width * 0.48,
         'vat_col_x': img_width * 0.60,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'center',
         'address_align': 'left',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]}
    ]

    store_name = fake.company()
    store_address = f"{fake.street_address()}, {fake.city()}, {fake.state()} - {fake.postcode()}"
    store_phone = fake.phone_number()
    gst_number = fake.bothify(text='GSTIN: ??????????????')
    receipt_id = fake.bothify(text='TRN: ##########')
    transaction_date = datetime.now().strftime("%d-%m-%Y")
    transaction_time = datetime.now().strftime("%H:%M:%S")

    all_vat_codes_map = {'A': 0.00, 'B': 0.05, 'C': 0.12, 'D': 0.18, 'E': 0.28}
    num_vat_codes_to_use = random.randint(1, 2)
    selected_vat_codes = random.sample(list(all_vat_codes_map.keys()), num_vat_codes_to_use)
    vat_rates_map = {code: all_vat_codes_map[code] for code in selected_vat_codes}

    num_items = random.randint(4, 8)
    items = []
    overall_gross_amount = 0.0
    vat_summary = {}
    for _ in range(num_items):
        item_name_length = random.choice([1, 2, 3, 5, 6, 7])
        item_name = fake.sentence(nb_words=item_name_length).replace('.', '').strip()
        unit_price = round(random.uniform(20.0, 300.0), 2)
        quantity = random.randint(1, 4)
        vat_code = random.choice(list(vat_rates_map.keys()))
        vat_percent = vat_rates_map[vat_code]
        item_net_amount = round(unit_price * quantity, 2)
        item_vat_amount = round(item_net_amount * vat_percent, 2)
        item_gross_amount = round(item_net_amount + item_vat_amount, 2)
        items.append({
            "name": item_name,
            "unit_price": unit_price,
            "qty": quantity,
            "vat_code": vat_code,
            "gross_amount": item_gross_amount
        })
        overall_gross_amount += item_gross_amount
        if vat_code not in vat_summary:
            vat_summary[vat_code] = {"net_amount": 0.0, "vat_amount": 0.0, "vat_percent": vat_percent}
        vat_summary[vat_code]["net_amount"] += item_net_amount
        vat_summary[vat_code]["vat_amount"] += item_vat_amount
    sorted_vat_summary_keys = sorted(vat_summary.keys())
    payment_method = random.choice(['CASH', 'UPI', 'CARD'])

    y_pos = 40
    line_spacing_tight = 30
    line_spacing_medium = 40
    line_spacing_large = 50
    current_layout = random.choice(layouts)

    def draw_centered_text(drawer, text, y, font, fill_color, current_img=img):
        bbox = drawer.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        x = (current_img.width - text_width) / 2
        drawer.text((x, y), text, font=font, fill=fill_color)

    def draw_aligned_text(drawer, text, y, font, alignment, fill_color, current_img=img):
        if alignment == 'left':
            drawer.text((padding, y), text, font=font, fill=fill_color)
        elif alignment == 'right':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox[0]
            x = current_img.width - padding - text_width
            drawer.text((x, y), text, font=font, fill=fill_color)
        elif alignment == 'center':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox[0]
            x = (current_img.width - text_width) / 2
            drawer.text((x, y), text, font=font, fill=fill_color)

    def wrap_text(text, font, max_width, draw_instance):
        lines = []
        words = text.split(' ')
        current_line = ""
        for word in words:
            test_line = f"{current_line} {word}".strip()
            text_width = draw_instance.textlength(test_line, font=font)
            if text_width <= max_width:
                current_line = test_line
            else:
                if current_line:
                    lines.append(current_line)
                current_line = word
                if draw_instance.textlength(word, font=font) > max_width:
                    lines.append(word)
                    current_line = ""
        if current_line:
            lines.append(current_line)
        return lines

    draw_aligned_text(draw, store_name.upper(), y_pos, font_large, current_layout['name_align'], text_color)
    y_pos += line_spacing_large + 10
    draw_aligned_text(draw, store_address, y_pos, font_small, current_layout['address_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, f"Ph: {store_phone}", y_pos, font_small, current_layout['phone_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, gst_number, y_pos, font_small, current_layout['gst_align'], text_color)
    y_pos += line_spacing_large + 10

    draw.text((padding, y_pos), f"TRN: {receipt_id}", font=font_small, fill=text_color)
    bbox = draw.textbbox((0, 0), f"{transaction_date} {transaction_time}", font=font_small)
    draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos),
              f"{transaction_date} {transaction_time}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15

    col_item_x = current_layout['item_col_x']
    col_qty_x = current_layout['qty_col_x']
    col_price_x = current_layout['price_col_x']
    col_vat_x = current_layout['vat_col_x']
    col_amount_x = current_layout['amount_col_x_func']()
    draw.text((col_item_x, y_pos), "ITEM", font=font_medium, fill=text_color)
    draw.text((col_qty_x, y_pos), "QTY", font=font_medium, fill=text_color)
    draw.text((col_price_x, y_pos), "PRICE", font=font_medium, fill=text_color)
    draw.text((col_vat_x, y_pos), "VAT", font=font_medium, fill=text_color)
    draw.text((col_amount_x, y_pos), "AMOUNT", font=font_medium, fill=text_color)
    header_bottom_y = y_pos + line_spacing_tight + 10
    draw.line([(padding, header_bottom_y), (img_width - padding, header_bottom_y)], fill=text_color, width=2)
    y_pos = header_bottom_y
    draw.line([(col_qty_x - 10, y_pos), (col_qty_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_price_x - 10, y_pos), (col_price_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_vat_x - 10, y_pos), (col_vat_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    item_max_width = col_qty_x - col_item_x - 10
    for item in items:
        y_pos += 5
        draw.line([(padding, y_pos), (img_width - padding, y_pos)], fill=text_color, width=1)
        y_pos += 5
        item_lines = wrap_text(item['name'].upper(), font_item_name, item_max_width, draw)
        for line in item_lines:
            draw.text((col_item_x, y_pos), line, font=font_item_name, fill=text_color)
            y_pos += line_spacing_tight
        y_pos -= line_spacing_tight
        draw.text((col_qty_x, y_pos), str(item['qty']), font=font_item_name, fill=text_color)
        draw.text((col_price_x, y_pos), f"{item['unit_price']:.2f}", font=font_item_name, fill=text_color)
        draw.text((col_vat_x, y_pos), item['vat_code'], font=font_item_name, fill=text_color)
        bbox = draw.textbbox((0, 0), f"{item['gross_amount']:.2f}", font=font_item_name)
        draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos),
                  f"{item['gross_amount']:.2f}", font=font_item_name, fill=text_color)
        y_pos += line_spacing_tight + 5
    y_pos += line_spacing_medium + 10

    total_amount_text = f"Rs. {overall_gross_amount:.2f}"
    bbox_total = draw.textbbox((0, 0), total_amount_text, font=font_total)
    total_amount_width = bbox_total[2] - bbox_total[0]
    draw.text((img_width - padding - total_amount_width, y_pos),
              total_amount_text, font=font_total, fill=text_color)
    draw.text((padding, y_pos), "TOTAL AMOUNT:", font=font_total, fill=text_color)
    y_pos += line_spacing_large + 20

    if sorted_vat_summary_keys:
        y_pos += line_spacing_medium
        draw_centered_text(draw, "VAT SUMMARY", y_pos, font_medium, text_color)
        y_pos += line_spacing_medium + 10
        col_vat_code_x = padding
        col_net_x = img_width * 0.30
        col_vat_percent_x = img_width * 0.55
        col_vat_amount_x = img_width - padding - draw.textbbox((0, 0), "VAT AMT", font_small)[2]
        draw.text((col_vat_code_x, y_pos), "VAT", font=font_small, fill=text_color)
        draw.text((col_net_x, y_pos), "NET", font=font_small, fill=text_color)
        draw.text((col_vat_percent_x, y_pos), "VAT%", font=font_small, fill=text_color)
        draw.text((col_vat_amount_x, y_pos), "VAT AMT", font=font_small, fill=text_color)
        y_pos += line_spacing_tight + 10
        for vat_code in sorted_vat_summary_keys:
            vat_info = vat_summary[vat_code]
            draw.text((col_vat_code_x, y_pos), vat_code, font=font_small, fill=text_color)
            draw.text((col_net_x, y_pos), f"{vat_info['net_amount']:.2f}", font=font_small, fill=text_color)
            draw.text((col_vat_percent_x, y_pos), f"{vat_info['vat_percent']*100:.0f}%", font=font_small, fill=text_color)
            bbox = draw.textbbox((0, 0), f"{vat_info['vat_amount']:.2f}", font=font_small)
            draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos),
                      f"{vat_info['vat_amount']:.2f}", font=font_small, fill=text_color)
            y_pos += line_spacing_tight
        y_pos += line_spacing_large

    draw.text((padding, y_pos), f"PAYMENT METHOD: {payment_method}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15
    thankyou_align = random.choice(['left', 'right', 'center'])
    draw_aligned_text(draw, "THANK YOU FOR YOUR PURCHASE!", y_pos, font_thankyou, thankyou_align, text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, "HAVE A GREAT DAY!", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_medium + 15
    draw_aligned_text(draw, "* NO REFUND ON SALE ITEMS *", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_large

    final_img = img.crop((0, 0, img_width, y_pos))
    final_img = add_paper_texture_and_wrinkles(final_img)
    final_img = add_scanner_noise(final_img)
    final_img = add_shadow_gradients(final_img)
    final_img = add_ink_smudges(final_img)
    final_img = add_scaling_stretching(final_img)
    final_img = add_random_cropping(final_img)
    final_img = add_gaussian_noise(final_img)
    final_img = add_salt_pepper_noise(final_img)
    final_img = add_speckle_noise(final_img)
    final_img = add_coffee_stains(final_img)
    final_img = add_folds(final_img)
    final_img = add_tears(final_img)
    final_img = apply_random_color_tint(final_img)

    # **No horizontal flipping here to avoid mirrored images**

    final_img.save(filename)
    print(f"Receipt image saved as {filename}")

def apply_random_color_tint(image):
    hsv_image = image.convert('HSV')
    hsv_pixels = hsv_image.load()
    tint_type = random.choice(['sepia', 'cool', 'warm', 'none'])
    if tint_type == 'sepia':
        hue_shift = random.randint(15, 30)
        saturation_factor = random.uniform(0.7, 0.9)
    elif tint_type == 'cool':
        hue_shift = random.randint(120, 180) 
        saturation_factor = random.uniform(0.8, 1.0)
    elif tint_type == 'warm':
        hue_shift = random.randint(30, 60)
        saturation_factor = random.uniform(0.8, 1.0)
    else:
        hue_shift = random.randint(0, 255)
        saturation_factor = random.uniform(0.9, 1.0)
    width, height = hsv_image.size
    for i in range(width):
        for j in range(height):
            h, s, v = hsv_pixels[i, j]
            h = (h + hue_shift) % 256
            s = int(s * saturation_factor)
            hsv_pixels[i, j] = (h, s, v)
    return hsv_image.convert('RGB')

# [All other add_* functions remain unchanged; include them here as before]

# Include definitions for:
# add_gaussian_noise, add_blur, add_tears, add_folds, add_paper_texture_and_wrinkles,
# add_scanner_noise, add_shadow_gradients, add_ink_smudges, add_scaling_stretching,
# add_random_cropping, add_salt_pepper_noise, add_speckle_noise, add_coffee_stains

if __name__ == "__main__":
    for i in range(10):
        generate_realistic_receipt_with_all_noises_and_effects(
            filename=os.path.join(output_folder, f"straight_receipt_{i+1}.png"),
            rotate_angle=0,
            perspective_perturb=0
        )
    print("Generated 10 straight, readable receipt images with realistic effects in 'receipts_straight' folder.")

# font changes 

In [ ]:
from faker import Faker
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import random
from datetime import datetime
import numpy as np
import os

output_folder = "receipts_straight2"
os.makedirs(output_folder, exist_ok=True)

# Add a list of font paths (readable and less readable fonts)
FONT_LIST = [
    "arial.ttf",      # Common, readable
    "arialbd.ttf",    # Bold, readable
    "impact.ttf",     # Less legible
    "comic.ttf",      # Comic Sans, less formal
    "times.ttf",      # Serif, classic
    "cour.ttf",       # Courier, monospaced
    # Add more or modify as needed
]

def pick_font(default, size=16):
    try:
        path = random.choice(FONT_LIST)
        return ImageFont.truetype(path, size)
    except IOError:
        return ImageFont.load_default()

def make_image_unreadable(image):
    # Apply strong blur
    image = image.filter(ImageFilter.GaussianBlur(radius=random.randint(6, 12)))
    # Add salt-and-pepper noise
    arr = np.array(image)
    noise_prob = 0.2  # 20% probability for pixels to be noisy
    mask = np.random.choice([0, 255], arr.shape[:2], p=[1 - noise_prob, noise_prob])
    arr[mask == 255] = 255  # Salt
    arr[mask == 0] = 0      # Pepper
    return Image.fromarray(arr)

def generate_realistic_receipt_with_all_noises_and_effects(filename="receipt.png", rotate_angle=0, perspective_perturb=0):
    fake = Faker('en_IN')

    img_width = 700
    img_height = 1500
    padding = 40
    background_color = (245, 245, 245)
    text_color = (30, 30, 30)
    img = Image.new('RGB', (img_width, img_height), color=background_color)
    draw = ImageDraw.Draw(img)

    # Random fonts for all sections
    font_xsmall = pick_font("arial.ttf", 17)
    font_small = pick_font("arial.ttf", 19)
    font_medium = pick_font("arial.ttf", 22)
    font_large = pick_font("arialbd.ttf", 32)
    font_item_name = pick_font("arial.ttf", 19)
    font_total = pick_font("arialbd.ttf", 28)
    font_thankyou = pick_font("arial.ttf", 22)

    layouts = [
        {'name_align': random.choice(['left', 'right', 'center']),
         'address_align': random.choice(['left', 'right', 'center']),
         'phone_align': random.choice(['left', 'right', 'center']),
         'gst_align': random.choice(['left', 'right', 'center']),
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'left',
         'address_align': 'left',
         'phone_align': 'left',
         'gst_align': 'left',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.45,
         'price_col_x': img_width * 0.58,
         'vat_col_x': img_width * 0.73,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'right',
         'address_align': 'right',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.35,
         'price_col_x': img_width * 0.48,
         'vat_col_x': img_width * 0.60,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'center',
         'address_align': 'left',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]}
    ]

    store_name = fake.company()
    store_address = f"{fake.street_address()}, {fake.city()}, {fake.state()} - {fake.postcode()}"
    store_phone = fake.phone_number()
    gst_number = fake.bothify(text='GSTIN: ??????????????')
    receipt_id = fake.bothify(text='TRN: ##########')
    transaction_date = datetime.now().strftime("%d-%m-%Y")
    transaction_time = datetime.now().strftime("%H:%M:%S")

    all_vat_codes_map = {'A': 0.00, 'B': 0.05, 'C': 0.12, 'D': 0.18, 'E': 0.28}
    num_vat_codes_to_use = random.randint(1, 2)
    selected_vat_codes = random.sample(list(all_vat_codes_map.keys()), num_vat_codes_to_use)
    vat_rates_map = {code: all_vat_codes_map[code] for code in selected_vat_codes}

    num_items = random.randint(4, 8)
    items = []
    overall_gross_amount = 0.0
    vat_summary = {}
    for _ in range(num_items):
        item_name_length = random.choice([1, 2, 3, 5, 6, 7])
        item_name = fake.sentence(nb_words=item_name_length).replace('.', '').strip()
        unit_price = round(random.uniform(20.0, 300.0), 2)
        quantity = random.randint(1, 4)
        vat_code = random.choice(list(vat_rates_map.keys()))
        vat_percent = vat_rates_map[vat_code]
        item_net_amount = round(unit_price * quantity, 2)
        item_vat_amount = round(item_net_amount * vat_percent, 2)
        item_gross_amount = round(item_net_amount + item_vat_amount, 2)
        items.append({
            "name": item_name,
            "unit_price": unit_price,
            "qty": quantity,
            "vat_code": vat_code,
            "gross_amount": item_gross_amount
        })
        overall_gross_amount += item_gross_amount
        if vat_code not in vat_summary:
            vat_summary[vat_code] = {"net_amount": 0.0, "vat_amount": 0.0, "vat_percent": vat_percent}
        vat_summary[vat_code]["net_amount"] += item_net_amount
        vat_summary[vat_code]["vat_amount"] += item_vat_amount
    sorted_vat_summary_keys = sorted(vat_summary.keys())
    payment_method = random.choice(['CASH', 'UPI', 'CARD'])

    y_pos = 40
    line_spacing_tight = 30
    line_spacing_medium = 40
    line_spacing_large = 50
    current_layout = random.choice(layouts)

    def draw_centered_text(drawer, text, y, font, fill_color, current_img=img):
        bbox = drawer.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox
        x = (current_img.width - text_width) / 2
        drawer.text((x, y), text, font=font, fill=fill_color)

    def draw_aligned_text(drawer, text, y, font, alignment, fill_color, current_img=img):
        if alignment == 'left':
            drawer.text((padding, y), text, font=font, fill=fill_color)
        elif alignment == 'right':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox
            x = current_img.width - padding - text_width
            drawer.text((x, y), text, font=font, fill=fill_color)
        elif alignment == 'center':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox
            x = (current_img.width - text_width) / 2
            drawer.text((x, y), text, font=font, fill=fill_color)

    def wrap_text(text, font, max_width, draw_instance):
        lines = []
        words = text.split(' ')
        current_line = ""
        for word in words:
            test_line = f"{current_line} {word}".strip()
            text_width = draw_instance.textlength(test_line, font=font)
            if text_width <= max_width:
                current_line = test_line
            else:
                if current_line:
                    lines.append(current_line)
                current_line = word
                if draw_instance.textlength(word, font=font) > max_width:
                    lines.append(word)
                    current_line = ""
        if current_line:
            lines.append(current_line)
        return lines

    draw_aligned_text(draw, store_name.upper(), y_pos, font_large, current_layout['name_align'], text_color)
    y_pos += line_spacing_large + 10
    draw_aligned_text(draw, store_address, y_pos, font_small, current_layout['address_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, f"Ph: {store_phone}", y_pos, font_small, current_layout['phone_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, gst_number, y_pos, font_small, current_layout['gst_align'], text_color)
    y_pos += line_spacing_large + 10

    draw.text((padding, y_pos), f"TRN: {receipt_id}", font=font_small, fill=text_color)
    bbox = draw.textbbox((0, 0), f"{transaction_date} {transaction_time}", font=font_small)
    draw.text((img_width - padding - (bbox[2] - bbox), y_pos),
              f"{transaction_date} {transaction_time}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15

    col_item_x = current_layout['item_col_x']
    col_qty_x = current_layout['qty_col_x']
    col_price_x = current_layout['price_col_x']
    col_vat_x = current_layout['vat_col_x']
    col_amount_x = current_layout['amount_col_x_func']()
    draw.text((col_item_x, y_pos), "ITEM", font=font_medium, fill=text_color)
    draw.text((col_qty_x, y_pos), "QTY", font=font_medium, fill=text_color)
    draw.text((col_price_x, y_pos), "PRICE", font=font_medium, fill=text_color)
    draw.text((col_vat_x, y_pos), "VAT", font=font_medium, fill=text_color)
    draw.text((col_amount_x, y_pos), "AMOUNT", font=font_medium, fill=text_color)
    header_bottom_y = y_pos + line_spacing_tight + 10
    draw.line([(padding, header_bottom_y), (img_width - padding, header_bottom_y)], fill=text_color, width=2)
    y_pos = header_bottom_y
    draw.line([(col_qty_x - 10, y_pos), (col_qty_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_price_x - 10, y_pos), (col_price_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_vat_x - 10, y_pos), (col_vat_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    item_max_width = col_qty_x - col_item_x - 10
    for item in items:
        y_pos += 5
        draw.line([(padding, y_pos), (img_width - padding, y_pos)], fill=text_color, width=1)
        y_pos += 5
        item_lines = wrap_text(item['name'].upper(), font_item_name, item_max_width, draw)
        for line in item_lines:
            draw.text((col_item_x, y_pos), line, font=font_item_name, fill=text_color)
            y_pos += line_spacing_tight
        y_pos -= line_spacing_tight
        draw.text((col_qty_x, y_pos), str(item['qty']), font=font_item_name, fill=text_color)
        draw.text((col_price_x, y_pos), f"{item['unit_price']:.2f}", font=font_item_name, fill=text_color)
        draw.text((col_vat_x, y_pos), item['vat_code'], font=font_item_name, fill=text_color)
        bbox = draw.textbbox((0, 0), f"{item['gross_amount']:.2f}", font=font_item_name)
        draw.text((img_width - padding - (bbox[2] - bbox), y_pos),
                  f"{item['gross_amount']:.2f}", font=font_item_name, fill=text_color)
        y_pos += line_spacing_tight + 5
        y_pos += line_spacing_medium + 10

    total_amount_text = f"Rs. {overall_gross_amount:.2f}"
    bbox_total = draw.textbbox((0, 0), total_amount_text, font=font_total)
    total_amount_width = bbox_total[2] - bbox_total
    draw.text((img_width - padding - total_amount_width, y_pos),
              total_amount_text, font=font_total, fill=text_color)
    draw.text((padding, y_pos), "TOTAL AMOUNT:", font=font_total, fill=text_color)
    y_pos += line_spacing_large + 20

    if sorted_vat_summary_keys:
        y_pos += line_spacing_medium
        draw_centered_text(draw, "VAT SUMMARY", y_pos, font_medium, text_color)
        y_pos += line_spacing_medium + 10
        col_vat_code_x = padding
        col_net_x = img_width * 0.30
        col_vat_percent_x = img_width * 0.55
        col_vat_amount_x = img_width - padding - draw.textbbox((0, 0), "VAT AMT", font_small)[2]
        draw.text((col_vat_code_x, y_pos), "VAT", font=font_small, fill=text_color)
        draw.text((col_net_x, y_pos), "NET", font=font_small, fill=text_color)
        draw.text((col_vat_percent_x, y_pos), "VAT%", font=font_small, fill=text_color)
        draw.text((col_vat_amount_x, y_pos), "VAT AMT", font=font_small, fill=text_color)
        y_pos += line_spacing_tight + 10
        for vat_code in sorted_vat_summary_keys:
            vat_info = vat_summary[vat_code]
            draw.text((col_vat_code_x, y_pos), vat_code, font=font_small, fill=text_color)
            draw.text((col_net_x, y_pos), f"{vat_info['net_amount']:.2f}", font=font_small, fill=text_color)
            draw.text((col_vat_percent_x, y_pos), f"{vat_info['vat_percent']*100:.0f}%", font=font_small, fill=text_color)
            bbox = draw.textbbox((0, 0), f"{vat_info['vat_amount']:.2f}", font=font_small)
            draw.text((img_width - padding - (bbox[2] - bbox), y_pos),
                      f"{vat_info['vat_amount']:.2f}", font=font_small, fill=text_color)
            y_pos += line_spacing_tight
        y_pos += line_spacing_large

    draw.text((padding, y_pos), f"PAYMENT METHOD: {payment_method}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15
    thankyou_align = random.choice(['left', 'right', 'center'])
    draw_aligned_text(draw, "THANK YOU FOR YOUR PURCHASE!", y_pos, font_thankyou, thankyou_align, text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, "HAVE A GREAT DAY!", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_medium + 15
    draw_aligned_text(draw, "* NO REFUND ON SALE ITEMS *", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_large

    final_img = img.crop((0, 0, img_width, y_pos))

    # Randomly make some images unreadable
    if random.random() < 0.3:  # ~30% chance
        final_img = make_image_unreadable(final_img)

    final_img.save(filename)
    print(f"Receipt image saved as {filename}")

if __name__ == "__main__":
    for i in range(10):
        generate_realistic_receipt_with_all_noises_and_effects(
            filename=os.path.join(output_folder, f"straight_receipt_{i+1}.png"),
            rotate_angle=0,
            perspective_perturb=0
        )
    print("Generated 10 straight, readable receipt images, some intentionally unreadable.")


# 50 generate

In [ ]:
from faker import Faker
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import random
from datetime import datetime
import numpy as np
import os

output_folder = "receipts_straight1"
os.makedirs(output_folder, exist_ok=True)

# List of font paths (readable and less readable)
FONT_LIST = [
    "arial.ttf",
    "arialbd.ttf",
    "impact.ttf",
    "comic.ttf",
    "times.ttf",
    "cour.ttf",
    # Add more font files as required
]

def pick_font(size=16):
    try:
        path = random.choice(FONT_LIST)
        return ImageFont.truetype(path, size)
    except IOError:
        return ImageFont.load_default()

def generate_realistic_receipt_with_all_noises_and_effects(filename="receipt.png"):
    fake = Faker('en_IN')

    img_width, img_height = 700, 1500
    padding = 40
    background_color = (245, 245, 245)
    text_color = (30, 30, 30)

    img = Image.new('RGB', (img_width, img_height), color=background_color)
    draw = ImageDraw.Draw(img)

    # Fonts
    font_xsmall = pick_font(17)
    font_small = pick_font(19)
    font_medium = pick_font(22)
    font_large = pick_font(32)
    font_item_name = pick_font(19)
    font_total = pick_font(28)
    font_thankyou = pick_font(22)

    layouts = [
        {'name_align': random.choice(['left', 'right', 'center']),
         'address_align': random.choice(['left', 'right', 'center']),
         'phone_align': random.choice(['left', 'right', 'center']),
         'gst_align': random.choice(['left', 'right', 'center']),
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'left',
         'address_align': 'left',
         'phone_align': 'left',
         'gst_align': 'left',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.45,
         'price_col_x': img_width * 0.58,
         'vat_col_x': img_width * 0.73,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'right',
         'address_align': 'right',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.35,
         'price_col_x': img_width * 0.48,
         'vat_col_x': img_width * 0.60,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]},
        {'name_align': 'center',
         'address_align': 'left',
         'phone_align': 'right',
         'gst_align': 'right',
         'item_col_x': padding,
         'qty_col_x': img_width * 0.40,
         'price_col_x': img_width * 0.52,
         'vat_col_x': img_width * 0.68,
         'amount_col_x_func': lambda: img_width - padding - draw.textbbox((0, 0), "AMOUNT", font_medium)[2]}
    ]

    store_name = fake.company()
    store_address = f"{fake.street_address()}, {fake.city()}, {fake.state()} - {fake.postcode()}"
    store_phone = fake.phone_number()
    gst_number = fake.bothify(text='GSTIN: ??????????????')
    receipt_id = fake.bothify(text='TRN: ##########')
    transaction_date = datetime.now().strftime("%d-%m-%Y")
    transaction_time = datetime.now().strftime("%H:%M:%S")

    all_vat_codes_map = {'A': 0.00, 'B': 0.05, 'C': 0.12, 'D': 0.18, 'E': 0.28}
    num_vat_codes_to_use = random.randint(1, 2)
    selected_vat_codes = random.sample(list(all_vat_codes_map.keys()), num_vat_codes_to_use)
    vat_rates_map = {code: all_vat_codes_map[code] for code in selected_vat_codes}

    num_items = random.randint(4, 8)
    items = []
    overall_gross_amount = 0.0
    vat_summary = {}

    for _ in range(num_items):
        item_name_length = random.choice([1, 2, 3, 5, 6, 7])
        item_name = fake.sentence(nb_words=item_name_length).replace('.', '').strip()
        unit_price = round(random.uniform(20.0, 300.0), 2)
        quantity = random.randint(1, 4)
        vat_code = random.choice(list(vat_rates_map.keys()))
        vat_percent = vat_rates_map[vat_code]
        item_net_amount = round(unit_price * quantity, 2)
        item_vat_amount = round(item_net_amount * vat_percent, 2)
        item_gross_amount = round(item_net_amount + item_vat_amount, 2)
        items.append({
            "name": item_name,
            "unit_price": unit_price,
            "qty": quantity,
            "vat_code": vat_code,
            "gross_amount": item_gross_amount
        })
        overall_gross_amount += item_gross_amount
        if vat_code not in vat_summary:
            vat_summary[vat_code] = {"net_amount": 0.0, "vat_amount": 0.0, "vat_percent": vat_percent}
        vat_summary[vat_code]["net_amount"] += item_net_amount
        vat_summary[vat_code]["vat_amount"] += item_vat_amount

    sorted_vat_summary_keys = sorted(vat_summary.keys())
    payment_method = random.choice(['CASH', 'UPI', 'CARD'])

    y_pos = 40
    line_spacing_tight = 30
    line_spacing_medium = 40
    line_spacing_large = 50
    current_layout = random.choice(layouts)

    def draw_centered_text(drawer, text, y, font, fill_color, current_img=img):
        bbox = drawer.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        x = (current_img.width - text_width) / 2
        drawer.text((x, y), text, font=font, fill=fill_color)

    def draw_aligned_text(drawer, text, y, font, alignment, fill_color, current_img=img):
        if alignment == 'left':
            drawer.text((padding, y), text, font=font, fill=fill_color)
        elif alignment == 'right':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox[0]
            x = current_img.width - padding - text_width
            drawer.text((x, y), text, font=font, fill=fill_color)
        elif alignment == 'center':
            bbox = drawer.textbbox((0, 0), text, font=font)
            text_width = bbox[2] - bbox[0]
            x = (current_img.width - text_width) / 2
            drawer.text((x, y), text, font=font, fill=fill_color)

    def wrap_text(text, font, max_width, draw_instance):
        lines = []
        words = text.split(' ')
        current_line = ""
        for word in words:
            test_line = f"{current_line} {word}".strip()
            text_width = draw_instance.textlength(test_line, font=font)
            if text_width <= max_width:
                current_line = test_line
            else:
                if current_line:
                    lines.append(current_line)
                current_line = word
                if draw_instance.textlength(word, font=font) > max_width:
                    lines.append(word)
                    current_line = ""
        if current_line:
            lines.append(current_line)
        return lines

    draw_aligned_text(draw, store_name.upper(), y_pos, font_large, current_layout['name_align'], text_color)
    y_pos += line_spacing_large + 10
    draw_aligned_text(draw, store_address, y_pos, font_small, current_layout['address_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, f"Ph: {store_phone}", y_pos, font_small, current_layout['phone_align'], text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, gst_number, y_pos, font_small, current_layout['gst_align'], text_color)
    y_pos += line_spacing_large + 10

    draw.text((padding, y_pos), f"TRN: {receipt_id}", font=font_small, fill=text_color)
    bbox = draw.textbbox((0, 0), f"{transaction_date} {transaction_time}", font=font_small)
    draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos),
              f"{transaction_date} {transaction_time}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15

    col_item_x = current_layout['item_col_x']
    col_qty_x = current_layout['qty_col_x']
    col_price_x = current_layout['price_col_x']
    col_vat_x = current_layout['vat_col_x']
    col_amount_x = current_layout['amount_col_x_func']()
    draw.text((col_item_x, y_pos), "ITEM", font=font_medium, fill=text_color)
    draw.text((col_qty_x, y_pos), "QTY", font=font_medium, fill=text_color)
    draw.text((col_price_x, y_pos), "PRICE", font=font_medium, fill=text_color)
    draw.text((col_vat_x, y_pos), "VAT", font=font_medium, fill=text_color)
    draw.text((col_amount_x, y_pos), "AMOUNT", font=font_medium, fill=text_color)
    header_bottom_y = y_pos + line_spacing_tight + 10
    draw.line([(padding, header_bottom_y), (img_width - padding, header_bottom_y)], fill=text_color, width=2)
    y_pos = header_bottom_y
    draw.line([(col_qty_x - 10, y_pos), (col_qty_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_price_x - 10, y_pos), (col_price_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    draw.line([(col_vat_x - 10, y_pos), (col_vat_x - 10, y_pos + len(items) * line_spacing_tight * 1.5)],
              fill=text_color, width=1)
    item_max_width = col_qty_x - col_item_x - 10
    for item in items:
        y_pos += 5
        draw.line([(padding, y_pos), (img_width - padding, y_pos)], fill=text_color, width=1)
        y_pos += 5
        item_lines = wrap_text(item['name'].upper(), font_item_name, item_max_width, draw)
        for line in item_lines:
            draw.text((col_item_x, y_pos), line, font=font_item_name, fill=text_color)
            y_pos += line_spacing_tight
        draw.text((col_qty_x, y_pos - line_spacing_tight), str(item['qty']), font=font_item_name, fill=text_color)
        draw.text((col_price_x, y_pos - line_spacing_tight), f"{item['unit_price']:.2f}", font=font_item_name, fill=text_color)
        draw.text((col_vat_x, y_pos - line_spacing_tight), item['vat_code'], font=font_item_name, fill=text_color)
        bbox = draw.textbbox((0, 0), f"{item['gross_amount']:.2f}", font=font_item_name)
        draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos - line_spacing_tight),
                  f"{item['gross_amount']:.2f}", font=font_item_name, fill=text_color)
        y_pos += 10

    total_amount_text = f"Rs. {overall_gross_amount:.2f}"
    bbox_total = draw.textbbox((0, 0), total_amount_text, font=font_total)
    total_amount_width = bbox_total[2] - bbox_total[0]
    draw.text((img_width - padding - total_amount_width, y_pos),
              total_amount_text, font=font_total, fill=text_color)
    draw.text((padding, y_pos), "TOTAL AMOUNT:", font=font_total, fill=text_color)
    y_pos += line_spacing_large + 20

    if sorted_vat_summary_keys:
        y_pos += line_spacing_medium
        draw_centered_text(draw, "VAT SUMMARY", y_pos, font_medium, text_color)
        y_pos += line_spacing_medium + 10
        col_vat_code_x = padding
        col_net_x = img_width * 0.30
        col_vat_percent_x = img_width * 0.55
        col_vat_amount_x = img_width - padding - draw.textbbox((0, 0), "VAT AMT", font_small)[2]
        draw.text((col_vat_code_x, y_pos), "VAT", font=font_small, fill=text_color)
        draw.text((col_net_x, y_pos), "NET", font=font_small, fill=text_color)
        draw.text((col_vat_percent_x, y_pos), "VAT%", font=font_small, fill=text_color)
        draw.text((col_vat_amount_x, y_pos), "VAT AMT", font=font_small, fill=text_color)
        y_pos += line_spacing_tight + 10
        for vat_code in sorted_vat_summary_keys:
            vat_info = vat_summary[vat_code]
            draw.text((col_vat_code_x, y_pos), vat_code, font=font_small, fill=text_color)
            draw.text((col_net_x, y_pos), f"{vat_info['net_amount']:.2f}", font=font_small, fill=text_color)
            draw.text((col_vat_percent_x, y_pos), f"{vat_info['vat_percent']*100:.0f}%", font=font_small, fill=text_color)
            bbox = draw.textbbox((0, 0), f"{vat_info['vat_amount']:.2f}", font=font_small)
            draw.text((img_width - padding - (bbox[2] - bbox[0]), y_pos),
                      f"{vat_info['vat_amount']:.2f}", font=font_small, fill=text_color)
            y_pos += line_spacing_tight
        y_pos += line_spacing_large

    draw.text((padding, y_pos), f"PAYMENT METHOD: {payment_method}", font=font_small, fill=text_color)
    y_pos += line_spacing_medium + 15
    thankyou_align = random.choice(['left', 'right', 'center'])
    draw_aligned_text(draw, "THANK YOU FOR YOUR PURCHASE!", y_pos, font_thankyou, thankyou_align, text_color)
    y_pos += line_spacing_tight
    draw_aligned_text(draw, "HAVE A GREAT DAY!", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_medium + 15
    draw_aligned_text(draw, "* NO REFUND ON SALE ITEMS *", y_pos, font_small, thankyou_align, text_color)
    y_pos += line_spacing_large

    final_img = img.crop((0, 0, img_width, y_pos))

    # Convert to grayscale for all images
    final_img = final_img.convert("L")

    # Apply subtle blur to ~30% images
    if random.random() < 0.3:
        final_img = final_img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    final_img.save(filename)
    print(f"Receipt image saved as {filename}")

if __name__ == "__main__":
    for i in range(50):
        generate_realistic_receipt_with_all_noises_and_effects(
            filename=os.path.join(output_folder, f"straight_receipt_{i + 1}.png")
        )
    print("Generated 50 grayscale receipt images with some subtly blurred outputs.")
